## Generación de la Vector Store

In [2]:
from src.utils import cargar_datos

# Importar datos
df = cargar_datos()

In [3]:
from src.rag import generate_vector_database

# Creación de la base de datos
generate_vector_database(df)

Total de descripciones a indexar: 114364


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Generando embeddings e indexando en ChromaDB usando métrica 'l2'...
Base de datos vectorial guardada en: /home/javier/UNI/PLN_Practica_Final/agente/vector_db


## Uso de RAG en pruebas

In [35]:
from src.rag import obtener_retriever
from src.config import PATH_VECTOR_DB


# Generar retriever
retriever = obtener_retriever(k = 5 , path=PATH_VECTOR_DB)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

In [ ]:
# Probar con un ejemplo
titulo = df.sample(1).title.iloc[0]
print(f"Buscando contexto para el título: '{titulo}'\n")

docs_recuperados = retriever.invoke(titulo)

for i, doc in enumerate(docs_recuperados):
    print(f"--- Documento Recuperado #{i+1} ---")
    print(f"ID Original de la fila: {doc.metadata.get('row_id')}")
    print(f"Título del registro indexado: {doc.metadata.get('original_title')} {doc.metadata.get('category')}")
    print(f"Descripción recuperada (Contexto):\n{doc.page_content}\n")

Buscando contexto para el título: 'General Mills posts loss but Wall St. firms fare well'

--- Documento Recuperado #1 ---
ID Original de la fila: 34483
Título del registro indexado: General Mills Profit Drops as Costs Rise Business
Descripción recuperada (Contexto):
CHICAGO (Reuters) - General Mills Inc. &lt;A HREF="http://www.investor.reuters.com/FullQuote.aspx?ticker=GIS.N target=/stocks/quickinfo/fullquote"&gt;GIS.N&lt;/A&gt; on Tuesday reported a steeper fall in quarterly earnings than Wall Street had expected due to higher costs of key food ingredients and promotional spending.

--- Documento Recuperado #2 ---
ID Original de la fila: 34722
Título del registro indexado: General Mills Profit Down 19 Percent Business
Descripción recuperada (Contexto):
LOS ANGELES (Reuters) - General Mills Inc. on Tuesday reported a steeper-than-expected drop in quarterly net income due to higher food ingredient costs and stepped-up spending on promotions.

--- Documento Recuperado #3 ---
ID Original

In [ ]:
from src.classifiers import predecir_categoria_rag, predecir_categoria

# Comparación de respuesta de LLM con RAG y sin RAG
titulo = "Lebron James will be president"

# FallBack en false para asegurar que el prompting es lo suficientemente bueno para que conteste como debe
print(f"Versión RAG:    {predecir_categoria_rag(titulo,retriever, fallback=False)}")
print(f"Versión no RAG: {predecir_categoria(titulo, fallback=False)}")

Versión RAG:    Business
Versión no RAG: World


## Comparativas de los métodos de predicción (LLM - LLM + RAG - KNN + RAG)

In [ ]:
df_test = cargar_datos(conjunto="test")
df_test

,label,title,description
0,3,Fears for T N pension after talks,Unions representing workers at Turner Newall s...
1,4,The Race is On: Second Private Team Sets Launc...,"SPACE.com - TORONTO, Canada -- A second team o..."
2,4,Ky. Company Wins Grant to Study Peptides (AP),AP - A company founded by a chemistry research...
3,4,Prediction Unit Helps Forecast Wildfires (AP),AP - It's barely dawn when Mike Fitzpatrick st...
4,4,Calif. Aims to Limit Farm-Related Smog (AP),AP - Southern California's smog-fighting agenc...
...,...,...,...
7594,1,Powell pushes diplomacy for N. Korea,WASHINGTON -- Outgoing Secretary of State Coli...
7596,2,Void is filled with Clement,With the supply of attractive pitching options...
7597,2,Martinez leaves bitter,Like Roger Clemens did almost exactly eight ye...
7598,3,5 of arthritis patients in Singapore take Bext...,SINGAPORE : Doctors in the United States have ...


In [ ]:
from tqdm import tqdm

from src.rag import obtener_retriever, generate_vector_database
from src.classifiers import predecir_categoria, predecir_categoria_rag, predecir_categoria_knn
from src.utils import fallback_postprocessing
from src.config import CLASS_TO_LABEL

In [ ]:
k = 5
espacios_distancia = ["cosine", "l2", "ip"]
retrievers = {}

# Crear los distintos retrievers
for space in espacios_distancia:
    print(f" ------ Configurando Vector Store para: {space} ------")
    
    vector_store = generate_vector_database(df, space=space, save=False)
    retriever = obtener_retriever(k, vector_store)
    
    retrievers[space] = retriever

print("Retrievers generados")
print(list(retrievers.keys()))

Configurando Vector Store para: cosine.
Total de descripciones a indexar: 114364


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Generando embeddings e indexando en ChromaDB usando métrica 'cosine'...
Base de datos vectorial generada en memoria (no persistente).
Configurando Vector Store para: l2.
Total de descripciones a indexar: 114364


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Generando embeddings e indexando en ChromaDB usando métrica 'l2'...
Base de datos vectorial generada en memoria (no persistente).
Configurando Vector Store para: ip.
Total de descripciones a indexar: 114364


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Generando embeddings e indexando en ChromaDB usando métrica 'ip'...
Base de datos vectorial generada en memoria (no persistente).
Retrievers generados
['cosine', 'l2', 'ip']


#### LLM

In [ ]:
y_true_no_rag = []
y_pred_no_rag = []

for _, row in tqdm(df_test.iterrows(), total=len(df_test), desc="Evaluando LLM sin RAG"):
    titulo = row["title"]
    label_real = row["label"]
    
    # Predicción
    pred_str = predecir_categoria(titulo, fallback=False)
    pred_str_fb = fallback_postprocessing(pred_str)
    
    # Conversión a ID (si no existe en las clases válidas, devuelve -1)
    pred_id = CLASS_TO_LABEL.get(pred_str_fb, -1)
    
    y_true_no_rag.append(label_real)
    y_pred_no_rag.append(pred_id)

Evaluando LLM sin RAG: 100%|██████████| 7569/7569 [11:37<00:00, 10.85it/s]


#### LLM + RAG

In [ ]:
y_true_rag = []
y_pred_rag = {"cosine": [], "l2": [], "ip": []}

for _, row in tqdm(df_test.iterrows(), total=len(df_test), desc="Evaluando LLM con RAG"):
    titulo = row["title"]
    label_real = row["label"]
    
    y_true_rag.append(label_real)
    
    for dist, retriever in retrievers.items():
        pred_str = predecir_categoria_rag(titulo, retriever, fallback=False)
        pred_str_fb = fallback_postprocessing(pred_str)
        
        pred_id = CLASS_TO_LABEL.get(pred_str_fb, -1)
        y_pred_rag[dist].append(pred_id)

Evaluando LLM con RAG: 100%|██████████| 7569/7569 [39:54<00:00,  3.16it/s]  


#### KNN

In [ ]:
y_true_knn = []
y_pred_knn = {"cosine": [], "l2": [], "ip": []}

for _, row in tqdm(df_test.iterrows(), total=len(df_test), desc="Evaluando KNN"):
    titulo = row["title"]
    label_real = row["label"]
    
    y_true_knn.append(label_real)
    
    for dist, retriever in retrievers.items():
        pred_str = predecir_categoria_knn(titulo, retriever)
        
        # Innecesario el -1, pero nunca sobra seguridad
        pred_id = CLASS_TO_LABEL.get(pred_str, -1)
        y_pred_knn[dist].append(pred_id)

Evaluando KNN: 100%|██████████| 7569/7569 [01:31<00:00, 82.69it/s]


#### Comparaciones

In [36]:
import pandas as pd

df_analysis = pd.DataFrame({
    "y_true": y_true_rag,
    "llm_no_rag": y_pred_no_rag,
    "llm_rag_cosine": y_pred_rag["cosine"],
    "llm_rag_l2": y_pred_rag["l2"],
    "llm_rag_ip": y_pred_rag["ip"],
    "knn_cosine": y_pred_knn["cosine"],
    "knn_l2": y_pred_knn["l2"],
    "knn_ip": y_pred_knn["ip"]
})

total_muestras = len(df_analysis)

In [37]:
# =======================================
# ANÁLISIS DE PREDICCIONES INVÁLIDAS (-1)
# =======================================
print("--- PREDICCIONES INVÁLIDAS (El modelo falló al dar una clase válida) ---")
invalids = (df_analysis == -1).sum()

print(f"LLM sin RAG:      {invalids['llm_no_rag']}")
print(f"LLM RAG (Cosine): {invalids['llm_rag_cosine']}")
print(f"LLM RAG (L2):     {invalids['llm_rag_l2']}")
print(f"LLM RAG (IP):     {invalids['llm_rag_ip']}")
print(f"KNN (Cosine):     {invalids['knn_cosine']}")
print(f"KNN (L2):         {invalids['knn_l2']}")
print(f"KNN (IP):         {invalids['knn_ip']}")

--- PREDICCIONES INVÁLIDAS (El modelo falló al dar una clase válida) ---
LLM sin RAG:      5
LLM RAG (Cosine): 39
LLM RAG (L2):     44
LLM RAG (IP):     44
KNN (Cosine):     0
KNN (L2):         0
KNN (IP):         0


In [ ]:
# ==============================================================
# DISCREPANCIAS INTRA-MODELO (Efecto de la métrica de distancia)
# ==============================================================
print("--- DISCREPANCIAS INTRA-MODELO (Mismo modelo, distinta distancia) ---")

# Dentro del LLM con RAG
diff_rag_cos_l2 = (df_analysis["llm_rag_cosine"] != df_analysis["llm_rag_l2"]).sum()
diff_rag_cos_ip = (df_analysis["llm_rag_cosine"] != df_analysis["llm_rag_ip"]).sum()
diff_rag_l2_ip  = (df_analysis["llm_rag_l2"]     != df_analysis["llm_rag_ip"]).sum()

# Dentro del KNN
diff_knn_cos_l2 = (df_analysis["knn_cosine"] != df_analysis["knn_l2"]).sum()
diff_knn_cos_ip = (df_analysis["knn_cosine"] != df_analysis["knn_ip"]).sum()
diff_knn_l2_ip =  (df_analysis["knn_l2"]     != df_analysis["knn_ip"]).sum()

print("En LLM con RAG:")
print(f"  Cosine vs L2: {diff_rag_cos_l2} diferencias ({(diff_rag_cos_l2/total_muestras)*100:.2f}%)")
print(f"  Cosine vs IP: {diff_rag_cos_ip} diferencias ({(diff_rag_cos_ip/total_muestras)*100:.2f}%)")
print(f"  L2 vs IP:     {diff_rag_l2_ip } diferencias ({(diff_rag_l2_ip /total_muestras)*100:.2f}%)")


print("\nEn KNN:")
print(f"  Cosine vs L2: {diff_knn_cos_l2} diferencias ({(diff_knn_cos_l2/total_muestras)*100:.2f}%)")
print(f"  Cosine vs IP: {diff_knn_cos_ip} diferencias ({(diff_knn_cos_ip/total_muestras)*100:.2f}%)")
print(f"  L2 vs IP:     {diff_knn_l2_ip } diferencias ({(diff_knn_l2_ip /total_muestras)*100:.2f}%)")

print()
print()

# ===============================================
# DISCREPANCIAS INTER-MODELO (LLM vs LLM con RAG)
# ===============================================
print("--- DISCREPANCIAS INTER-MODELO (LLM vs LLM+RAG) ---")

diff_norag_rag_cos = (df_analysis["llm_no_rag"] != df_analysis["llm_rag_cosine"]).sum()
diff_norag_rag_l2  = (df_analysis["llm_no_rag"] != df_analysis["llm_rag_l2"]).sum()
diff_norag_rag_ip  = (df_analysis["llm_no_rag"] != df_analysis["llm_rag_ip"]).sum()

print(f"LLM vs LLM+RAG (Cosine): {diff_norag_rag_cos} diferencias ({(diff_norag_rag_cos/total_muestras)*100:.2f}%)")
print(f"LLM vs LLM+RAG (L2):     {diff_norag_rag_l2} diferencias ({(diff_norag_rag_l2/total_muestras)*100:.2f}%)")
print(f"LLM vs LLM+RAG (IP):     {diff_norag_rag_ip} diferencias ({(diff_norag_rag_ip/total_muestras)*100:.2f}%)")

--- DISCREPANCIAS INTRA-MODELO (Mismo modelo, distinta distancia) ---
En LLM con RAG:
  Cosine vs L2: 27 diferencias (0.36%)
  Cosine vs IP: 27 diferencias (0.36%)
  L2 vs IP:     0 diferencias (0.00%)

En KNN:
  Cosine vs L2: 0 diferencias (0.00%)
  Cosine vs IP: 0 diferencias (0.00%)
  L2 vs IP:     0 diferencias (0.00%)


--- DISCREPANCIAS INTER-MODELO (LLM vs LLM+RAG) ---
LLM vs LLM+RAG (Cosine): 1772 diferencias (23.41%)
LLM vs LLM+RAG (L2):     1772 diferencias (23.41%)
LLM vs LLM+RAG (IP):     1772 diferencias (23.41%)


In [ ]:
# ===============================
# ANÁLISIS DE ACIERTOS (Accuracy)
# ===============================
print("--- ACIERTOS Y PRECISIÓN GLOBAL ---")

columnas_modelos = [
    "llm_no_rag", 
    "llm_rag_cosine", "llm_rag_l2", "llm_rag_ip",
    "knn_cosine", "knn_l2", "knn_ip"
]

for modelo in columnas_modelos:
    aciertos = (df_analysis[modelo] == df_analysis["y_true"]).sum()
    accuracy = (aciertos / total_muestras) * 100
    
    print(f"{modelo.ljust(15)}: {aciertos}/{total_muestras} aciertos ({accuracy:.2f}%)")


--- ACIERTOS Y PRECISIÓN GLOBAL ---
llm_no_rag     : 5875/7569 aciertos (77.62%)
llm_rag_cosine : 6324/7569 aciertos (83.55%)
llm_rag_l2     : 6319/7569 aciertos (83.49%)
llm_rag_ip     : 6319/7569 aciertos (83.49%)
knn_cosine     : 6101/7569 aciertos (80.61%)
knn_l2         : 6101/7569 aciertos (80.61%)
knn_ip         : 6101/7569 aciertos (80.61%)


In [40]:
casos_mejorados = df_analysis[
    (df_analysis["llm_no_rag"] != df_analysis["y_true"]) & 
    (df_analysis["llm_rag_cosine"] == df_analysis["y_true"])
]
print(f"El RAG arregló {len(casos_mejorados)} errores del LLM puro.")

El RAG arregló 1028 errores del LLM puro.
